In [0]:
"""
ETL Extract Layer — S&P 500 Stock Data Downloader & Encryptor
=============================================================
Downloads 5 years of daily OHLCV data for all S&P 500 tickers via yfinance,
saves each as an encrypted .csv.enc file using Fernet symmetric encryption.
 
Designed as the Bronze layer for a Databricks ETL pipeline.
The companion decrypt script (or your Databricks notebook) uses the saved
secret.key file to decrypt each file before loading into the Silver schema.
 
Usage:
    pip install yfinance cryptography pandas requests lxml
    python stock_etl_extract.py
 
Output structure:
    output/
    ├── secret.key          ← Keep this safe! Required for decryption.
    ├── encrypted/
    │   ├── AAPL.csv.enc
    │   ├── MSFT.csv.enc
    │   └── ...
    └── logs/
        └── run_<timestamp>.log
"""

In [0]:
import os
import time
import logging
import requests
import pandas as pd
import yfinance as yf
from pathlib import Path
from datetime import datetime, timedelta
from cryptography.fernet import Fernet

In [0]:
# ─── Configuration ────────────────────────────────────────────────────────────
 
OUTPUT_DIR       = Path("/Volumes/stocks/warehouse/raw_data/input")
ENCRYPTED_DIR    = OUTPUT_DIR / "encrypted"
LOG_DIR          = OUTPUT_DIR / "logs"
KEY_FILE         = OUTPUT_DIR / "secret.key"
 
YEARS_OF_DATA    = 4
BATCH_SIZE       = 50       # Tickers per yfinance batch download
RETRY_ATTEMPTS   = 3         # Retries per failed ticker
RETRY_DELAY      = 5         # Seconds between retries

In [0]:
# ─── Setup ────────────────────────────────────────────────────────────────────
 
def setup_directories():
    for d in [OUTPUT_DIR, ENCRYPTED_DIR, LOG_DIR]:
        d.mkdir(parents=True, exist_ok=True)
 
 
def setup_logging() -> logging.Logger:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file  = LOG_DIR / f"run_{timestamp}.log"
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

In [0]:
# ─── Encryption ───────────────────────────────────────────────────────────────
 
def load_or_create_key() -> Fernet:
    """Load existing Fernet key or generate a new one and persist it."""
    if KEY_FILE.exists():
        key = KEY_FILE.read_bytes()
        logging.info(f"Loaded existing encryption key from {KEY_FILE}")
    else:
        key = Fernet.generate_key()
        KEY_FILE.write_bytes(key)
        logging.info(f"Generated new encryption key → {KEY_FILE}")
    return Fernet(key)
 
 
def encrypt_and_save(fernet: Fernet, df: pd.DataFrame, ticker: str):
    """Serialize DataFrame to CSV bytes, encrypt, and write to disk."""
    csv_bytes       = df.to_csv(index=True).encode("utf-8")
    encrypted_bytes = fernet.encrypt(csv_bytes)
    out_path        = ENCRYPTED_DIR / f"{ticker}.csv.enc"
    out_path.write_bytes(encrypted_bytes)
    return out_path

In [0]:
# ─── Data Download ────────────────────────────────────────────────────────────
 
def get_sp500_tickers() -> list[str]:
    """Scrape the S&P 500 constituent list from Wikipedia."""
    logging.info("Fetching S&P 500 ticker list from Wikipedia...")
    url     = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0 (compatible; StockETL/1.0; +https://github.com)"}
    resp    = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    tables  = pd.read_html(resp.text)
    df      = tables[0]
    tickers = df["Symbol"].str.replace(".", "-", regex=False).tolist()
    logging.info(f"Found {len(tickers)} tickers.")
    return tickers
 
 
def download_batch(tickers: list[str], start: str, end: str) -> dict[str, pd.DataFrame]:
    """Download a batch of tickers in one yfinance call for efficiency."""
    joined = " ".join(tickers)
    raw    = yf.download(
        joined,
        start=start,
        end=end,
        group_by="ticker",
        auto_adjust=True,
        progress=False,
        threads=True,
    )
 
    results = {}
 
    if len(tickers) == 1:
        # Single ticker — yfinance returns a flat DataFrame
        ticker = tickers[0]
        if not raw.empty:
            raw.index.name = "Date"
            results[ticker] = raw
    else:
        for ticker in tickers:
            try:
                df = raw[ticker].dropna(how="all")
                if not df.empty:
                    df.index.name = "Date"
                    results[ticker] = df
            except KeyError:
                pass  # Ticker missing from batch response — handled as failed
 
    return results

In [0]:
# ─── Main Pipeline ────────────────────────────────────────────────────────────
 
def run():
    setup_directories()
    logger = setup_logging()
    logger.info("=" * 60)
    logger.info("  ETL Extract — S&P 500 Bronze Layer")
    logger.info("=" * 60)
 
    fernet  = load_or_create_key()
    tickers = get_sp500_tickers()
 
    end_date   = datetime.today()
    start_date = end_date - timedelta(days=365 * YEARS_OF_DATA)
    start_str  = start_date.strftime("%Y-%m-%d")
    end_str    = end_date.strftime("%Y-%m-%d")
    logger.info(f"Date range: {start_str} → {end_str}")
 
    # Track outcomes
    success, failed, skipped = [], [], []
 
    batches = [tickers[i:i+BATCH_SIZE] for i in range(0, len(tickers), BATCH_SIZE)]
    total_batches = len(batches)
 
    for batch_idx, batch in enumerate(batches, 1):
        logger.info(f"Batch {batch_idx}/{total_batches}: downloading {len(batch)} tickers...")
 
        for attempt in range(1, RETRY_ATTEMPTS + 1):
            try:
                batch_data = download_batch(batch, start_str, end_str)
                break
            except Exception as e:
                logger.warning(f"  Batch {batch_idx} attempt {attempt} failed: {e}")
                if attempt < RETRY_ATTEMPTS:
                    time.sleep(RETRY_DELAY)
                else:
                    logger.error(f"  Batch {batch_idx} permanently failed after {RETRY_ATTEMPTS} attempts.")
                    failed.extend(batch)
                    batch_data = {}
 
        for ticker in batch:
            # Skip if already encrypted from a previous run
            out_path = ENCRYPTED_DIR / f"{ticker}.csv.enc"
            if out_path.exists():
                logger.info(f"  [SKIP]    {ticker} — already exists.")
                skipped.append(ticker)
                continue
 
            df = batch_data.get(ticker)
            if df is None or df.empty:
                logger.warning(f"  [MISSING] {ticker} — no data returned.")
                failed.append(ticker)
                continue
 
            try:
                path = encrypt_and_save(fernet, df, ticker)
                size_kb = path.stat().st_size / 1024
                logger.info(f"  [OK]      {ticker} — {len(df)} rows → {path.name} ({size_kb:.1f} KB)")
                success.append(ticker)
            except Exception as e:
                logger.error(f"  [ERROR]   {ticker} — encryption failed: {e}")
                failed.append(ticker)
 
    # ─── Summary ──────────────────────────────────────────────────────────────
    logger.info("")
    logger.info("=" * 60)
    logger.info("  Run Summary")
    logger.info("=" * 60)
    logger.info(f"  Total tickers : {len(tickers)}")
    logger.info(f"  ✅ Succeeded  : {len(success)}")
    logger.info(f"  ⏭  Skipped    : {len(skipped)}")
    logger.info(f"  ❌ Failed     : {len(failed)}")
    if failed:
        logger.info(f"  Failed list  : {', '.join(failed)}")
    logger.info(f"  Output dir   : {ENCRYPTED_DIR.resolve()}")
    logger.info(f"  Secret key   : {KEY_FILE.resolve()}")
    logger.info("=" * 60)
    logger.info("  ⚠  Store secret.key securely — back it up to a vault")
    logger.info("     (e.g. Databricks Secrets, AWS Secrets Manager, Azure Key Vault)")
    logger.info("=" * 60)
 

In [0]:
run()